# Workforce Optimization with NVIDIA cuOpt

This notebook runs the workforce optimization using cuOpt on serverless GPU compute.
Results are written to Unity Catalog Delta tables.

In [0]:
%pip install -U nvidia-nccl-cu12
%pip install -q --extra-index-url=https://pypi.nvidia.com cuopt-server-cu12 cuopt-sh-client cuopt-cu12==25.8.*
%restart_python

In [0]:
import json
import subprocess
from datetime import datetime

# ── GPU check (fail fast before loading data or building the problem) ──
def check_gpu():
    try:
        output = subprocess.check_output(
            ["nvidia-smi"], shell=False, stderr=subprocess.STDOUT
        ).decode()
        print("GPU detected:")
        for line in output.split("\n")[:10]:
            print(f"  {line}")
        return True
    except Exception as e:
        print(f"No GPU detected: {e}")
        return False

has_gpu = check_gpu()
if not has_gpu:
    raise RuntimeError(
        "This notebook requires GPU compute. "
        "Please run on serverless GPU or a GPU cluster."
    )

# ── Parameters ──
# config_json is set as base_parameters on the job (from config)
# run_id and results_table are passed at run time via job_parameters
dbutils.widgets.text("config_json", '''{"source_catalog": "main", "source_schema": "optimization", "workers_table": "workers", "shifts_table": "shifts", "availability_table": "availability", "worker_name_col": "worker_name", "worker_pay_col": "pay_rate", "shift_name_col": "shift_name", "shift_required_col": "required_workers", "availability_worker_col": "worker_name", "availability_shift_col": "shift_name", "max_shifts_per_worker": 10, "time_limit_seconds": 60.0}''')
dbutils.widgets.text("run_id", "fb75ea41-94a1-46fb-9a00-cc51bde9c684")
dbutils.widgets.text("results_table", "main.staffing_optimization.optimization_results")
dbutils.widgets.text("owner_user", "")

config_json = dbutils.widgets.get("config_json")
run_id = dbutils.widgets.get("run_id")
results_table = dbutils.widgets.get("results_table")
owner_user = dbutils.widgets.get("owner_user") or None

# Parse configuration
config = json.loads(config_json)
print(f"Running optimization for run_id: {run_id}")
print(f"Results will be written to: {results_table}")

In [0]:
# spark is pre-initialized in Databricks notebooks
# Build table paths
source_catalog = config["source_catalog"]
source_schema = config["source_schema"]
workers_table = f"{source_catalog}.{source_schema}.{config['workers_table']}"
shifts_table = f"{source_catalog}.{source_schema}.{config['shifts_table']}"
availability_table = f"{source_catalog}.{source_schema}.{config['availability_table']}"

print(f"Loading data from:")
print(f"  Workers: {workers_table}")
print(f"  Shifts: {shifts_table}")
print(f"  Availability: {availability_table}")

# Get column names from config
worker_name_col = config.get("worker_name_col", "worker_name")
worker_pay_col = config.get("worker_pay_col", "pay_rate")
shift_name_col = config.get("shift_name_col", "shift_name")
shift_required_col = config.get("shift_required_col", "required_workers")
availability_worker_col = config.get("availability_worker_col", "worker_name")
availability_shift_col = config.get("availability_shift_col", "shift_name")

# Read Spark DataFrames first, then do Spark-compatible filtering/dedup before toPandas()
workers_sdf = spark.read.table(workers_table).select(worker_name_col, worker_pay_col).dropna()
shifts_sdf = spark.read.table(shifts_table).select(shift_name_col, shift_required_col).dropna()
availability_sdf = spark.read.table(availability_table).select(availability_worker_col, availability_shift_col)

# Keep only valid edges (worker exists, shift exists), and deduplicate in Spark
valid_workers_sdf = workers_sdf.selectExpr(f"{worker_name_col} as {availability_worker_col}").distinct()
valid_shifts_sdf = shifts_sdf.selectExpr(f"{shift_name_col} as {availability_shift_col}").distinct()

availability_filtered_sdf = (
    availability_sdf.dropna()
    .dropDuplicates([availability_worker_col, availability_shift_col])
    .join(valid_workers_sdf, on=availability_worker_col, how="inner")
    .join(valid_shifts_sdf, on=availability_shift_col, how="inner")
)

# Convert reduced data to pandas in memory for cuOpt model building
workers_df = workers_sdf.toPandas()
shifts_df = shifts_sdf.toPandas()
avail = availability_filtered_sdf.toPandas()

# Convert to dictionaries for optimization
worker_pay = dict(zip(workers_df[worker_name_col], workers_df[worker_pay_col]))
shift_requirements = dict(zip(shifts_df[shift_name_col], shifts_df[shift_required_col]))
availability = (
    avail.groupby(availability_worker_col)[availability_shift_col]
    .apply(list)
    .to_dict()
)

print(f"\nData summary:")
print(f"  Workers: {len(worker_pay)}")
print(f"  Shifts: {len(shift_requirements)}")
print(f"  Availability records: {len(avail)}")

In [0]:
# GPU check moved to Cell 2 (fail fast before data loading)

In [0]:
import numpy as np
import time
from collections import defaultdict

from cuopt.linear_programming.problem import Problem, VType, sense, LinearExpression
from cuopt.linear_programming.solver_settings import SolverSettings

# Create the optimization problem
problem = Problem("workforce_optimization")

# Add binary decision variables for each available (worker, shift) assignment
# and build adjacency indexes in one pass.
assignment_vars = {}
vars_by_shift = defaultdict(list)   # shift -> list[(var, cost)]
vars_by_worker = defaultdict(list)  # worker -> list[var]
all_vars = []
all_costs = []

for worker, shifts in availability.items():
    cost = float(worker_pay[worker])
    for shift in shifts:
        var = problem.addVariable(vtype=VType.INTEGER, lb=0.0, ub=1.0)
        assignment_vars[(worker, shift)] = var
        vars_by_shift[shift].append((var, cost))
        vars_by_worker[worker].append(var)

        if cost != 0:
            all_vars.append(var)
            all_costs.append(cost)

print(f"Created {len(assignment_vars)} binary decision variables")

# Create objective function: minimize total labor cost
problem.setObjective(LinearExpression(all_vars, all_costs, 0.0), sense.MINIMIZE)
print("Objective function set: minimize total labor cost")

# Add constraints: assign exactly the required number of workers to each shift
constraint_count = 0
for shift, required_count in shift_requirements.items():
    shift_pairs = vars_by_shift.get(shift, [])
    if shift_pairs:
        shift_vars = [v for v, _ in shift_pairs]
        problem.addConstraint(
            LinearExpression(shift_vars, [1.0] * len(shift_vars), 0.0) == required_count,
            name=f"shift_{shift}"
        )
        constraint_count += 1

print(f"Added {constraint_count} shift requirement constraints")

# Optional: Add max shifts per worker constraint
max_shifts = config.get("max_shifts_per_worker")
if max_shifts:
    for worker, worker_vars in vars_by_worker.items():
        if worker_vars:
            problem.addConstraint(
                LinearExpression(worker_vars, [1.0] * len(worker_vars), 0.0) <= max_shifts,
                name=f"max_shifts_{worker}"
            )
    print(f"Added max shifts constraint: {max_shifts} shifts per worker")

In [0]:
import os
print("CUDA_HOME:", os.environ.get("CUDA_HOME"))
print("LD_LIBRARY_PATH:", os.environ.get("LD_LIBRARY_PATH","")[:400], "...")


In [0]:
# Configure solver settings
time_limit = config.get("time_limit_seconds", 60.0)
settings = SolverSettings()
settings.set_parameter("time_limit", time_limit)
settings.set_parameter("log_to_console", True)
settings.set_parameter("method", 0)

print(f"\nSolver configured:")
print(f"  Problem type: {'MIP' if problem.IsMIP else 'LP'}")
print(f"  Variables: {problem.NumVariables}")
print(f"  Constraints: {problem.NumConstraints}")
print(f"  Time limit: {time_limit}s")

# Solve
print("\nSolving...")
start_time = time.time()
problem.solve(settings)
solve_time = time.time() - start_time

print(f"\nSolve completed in {solve_time:.3f} seconds")
print(f"Solver status: {problem.Status.name}")
print(f"Objective value: ${problem.ObjValue:.2f}")

In [0]:
# Check if optimization was successful
solver_status = problem.Status.name
print(f"Solver status: {solver_status}")

if solver_status == "Infeasible":
    # Problem is infeasible - no solution exists
    error_output = {
        "status": "infeasible",
        "error_type": "infeasible",
        "run_id": run_id,
        "message": "No feasible solution exists for this optimization problem.",
        "details": (
            "The constraints cannot be satisfied simultaneously. This often happens when:\n"
            "• max_shifts_per_worker is too restrictive\n"
            "• Workers don't have enough availability to cover shift requirements\n"
            "• Required workers per shift exceeds available workers\n\n"
            "Try adjusting your configuration:\n"
            "1. Increase or remove the max_shifts_per_worker constraint\n"
            "2. Add more workers or increase worker availability\n"
            "3. Reduce required_workers for some shifts"
        ),
        "solve_time_seconds": float(solve_time),
        "configuration": {
            "max_shifts_per_worker": config.get("max_shifts_per_worker"),
            "num_workers": len(worker_pay),
            "num_shifts": len(shift_requirements),
            "total_shift_requirements": sum(shift_requirements.values())
        }
    }
    print(json.dumps(error_output, indent=2))
    dbutils.notebook.exit(json.dumps(error_output))

elif solver_status not in ["Optimal", "FeasibleFound"]:
    # Other solver errors (timeout, numeric issues, etc.)
    error_output = {
        "status": "solver_error",
        "error_type": "solver_error",
        "run_id": run_id,
        "message": f"Solver terminated with status: {solver_status}",
        "details": "The optimization solver encountered an issue. Please check the configuration and try again.",
        "solve_time_seconds": float(solve_time),
        "solver_status": solver_status
    }
    print(json.dumps(error_output, indent=2))
    dbutils.notebook.exit(json.dumps(error_output))

# Extract the optimal solution
import pandas as pd


def extract_solution():
    """Extract the optimal solution."""
    assignments = []

    for (worker, shift), var in assignment_vars.items():
        if var.getValue() > 0.5:  # Binary variable is 1
            assignments.append({
                "worker_name": worker,
                "shift_name": shift,
                "cost": float(worker_pay[worker])
            })

    if assignments:
        results_tmp = pd.DataFrame(assignments)
        shift_assignments = results_tmp.groupby("shift_name")["worker_name"].apply(list).to_dict()
        worker_assignments = results_tmp.groupby("worker_name")["shift_name"].apply(list).to_dict()
    else:
        shift_assignments = {}
        worker_assignments = {}

    return assignments, shift_assignments, worker_assignments


assignments, shift_assignments, worker_assignments = extract_solution()

# Display solution summary
print(f"\n{'='*60}")
print("OPTIMIZATION RESULTS")
print(f"{'='*60}")
print(f"Total Labor Cost: ${problem.ObjValue:.2f}")
print(f"Solve Time: {solve_time:.3f} seconds")
print(f"Workers Assigned: {len(worker_assignments)}")
print(f"Shifts Covered: {len(shift_assignments)}")

print(f"\n{'='*60}")
print("SHIFT ASSIGNMENTS")
print(f"{'='*60}")
for shift in sorted(shift_assignments.keys()):
    workers = shift_assignments[shift]
    required = shift_requirements.get(shift, 0)
    total_cost = sum(worker_pay[w] for w in workers)
    print(f"  {shift}: {workers} (Required: {required}, Assigned: {len(workers)}, Cost: ${total_cost})")

print(f"\n{'='*60}")
print("WORKER ASSIGNMENTS")
print(f"{'='*60}")
for worker in sorted(worker_assignments.keys()):
    shifts = worker_assignments[worker]
    total_cost = len(shifts) * worker_pay[worker]
    print(f"  {worker}: {shifts} ({len(shifts)} shifts, ${total_cost})")

In [0]:
import pandas as pd

# Create results DataFrame with run_id column for the unified table
results_df = pd.DataFrame(assignments)
results_df["run_id"] = run_id
results_df["owner_user"] = owner_user
results_df["created_at"] = datetime.utcnow().isoformat()

# Display results
display(spark.createDataFrame(results_df))

# Save to unified Delta table (append mode - multiple runs in one table)
if results_table:
    print(f"\nSaving {len(results_df)} results to {results_table}")
    spark_results = spark.createDataFrame(results_df)
    
    # Check if table exists - if not, create it; if yes, append
    try:
        # First delete any existing results for this run_id (in case of re-run)
        spark.sql(f"DELETE FROM {results_table} WHERE run_id = '{run_id}'")
        print(f"Cleared any existing results for run_id: {run_id}")
    except Exception as e:
        # Table might not exist yet
        print(f"Table may be new or empty: {e}")
    
    # Write results (append to existing table or create new)
    spark_results.write.mode("append").option("mergeSchema", "true").saveAsTable(results_table)
    print(f"Results saved to {results_table}")
else:
    print("No results_table specified, skipping save")

In [0]:
# Return final summary for run output
output = {
    "status": "success",
    "run_id": run_id,
    "total_cost": float(problem.ObjValue),
    "solve_time_seconds": float(solve_time),
    "num_assignments": len(assignments),
    "num_workers_assigned": len(worker_assignments),
    "num_shifts_covered": len(shift_assignments),
    "results_table": results_table
}

print(f"\nRun completed successfully!")
print(json.dumps(output, indent=2))

dbutils.notebook.exit(json.dumps(output))